In [2]:
import pandas as pd

In [4]:
patient_df = pd.read_csv("C:\\Users\\USER 2\\Downloads\\Patient_Data.csv")
billing_df = pd.read_csv("C:\\Users\\USER 2\\Downloads\\Billing_Data.csv")
print("Patient Dataset Info:")
print(patient_df.info())
print("\nBilling Dataset Info:")
print(billing_df.info())

Patient Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes
None

Billing Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 252.0 bytes
None


In [10]:
import pandas as pd

# Select only relevant columns for billing
billing_relevant = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']].copy()

# Drop administrative columns
patient_cleaned = patient_df.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')

# Groupby to find total bill amount per department
dept_total = billing_relevant.groupby('Department')['BillAmount'].sum().reset_index()
print("\nTotal Bill Amount per Department:")
print(dept_total)

# Remove duplicate patient records based on PatientID
patient_cleaned = patient_cleaned.drop_duplicates(subset=['PatientID'])

# Fill missing BillAmount values with mean bill amount
mean_bill = billing_relevant['BillAmount'].mean()
billing_relevant.loc[:, 'BillAmount'] = billing_relevant['BillAmount'].fillna(mean_bill)

# Merge billing dataset with patient dataset on PatientID
merged_df = pd.merge(patient_cleaned, billing_df, on='PatientID', how='outer')
# Concatenate additional DataFrame (new patients for current week)
new_patients = pd.DataFrame({
    'PatientID': [201, 202],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Smith', 'Dr. Adams'],
    'BillAmount': [5000, 7000]
})
merged_df = pd.concat([merged_df, new_patients], axis=0)

# Concatenate new billing category columns (column-wise)
insurance_flags = [True, False] * (len(merged_df) // 2 + 1)
insurance_flags = insurance_flags[:len(merged_df)]  # ensure exact length

new_columns = pd.DataFrame({
    'InsuranceCovered': insurance_flags,
    'FinalAmount': merged_df['BillAmount'] * 0.8  # assuming 20% insurance coverage
})

merged_df = pd.concat(
    [merged_df.reset_index(drop=True),
     new_columns.reset_index(drop=True)],
    axis=1
)

# Final cleaned dataset
print("\nFinal Cleaned Dataset:")
print(merged_df.head())



Total Bill Amount per Department:
    Department  BillAmount
0   Cardiology     16200.0
1  Dermatology         0.0
2    Neurology         0.0
3  Orthopedics      7500.0

Final Cleaned Dataset:
   PatientID     Name   Department     Doctor  BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith      5000.0            2000.0   
1        102      Bob    Neurology   Dr. John         NaN            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0            2500.0   
3        104    David   Cardiology  Dr. Smith      6200.0            3000.0   
4        105      Eva  Dermatology   Dr. Rose         NaN            1000.0   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0              True       4000.0  
1       3500.0             False          NaN  
2       5000.0              True       6000.0  
3       3200.0             False       4960.0  
4       4000.0              True          NaN  
